# 🏛️ PROJECT 17: AMERICAN EXPRESS DEFAULT PREDICTION
> **Industrial Credit Risk & Financial Machine Learning Pipeline**

<p align="center">
  <img src="https://img.shields.io/badge/Status-Project%2017%20of%2020-00ffcc?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Architecture-Advanced%20Ensemble%20%2F%20DNN-ff007f?style=for-the-badge" />
  <img src="https://img.shields.io/badge/Domain-Financial%20Risk%20Management-00bfff?style=for-the-badge" />
</p>

## 🎯 The Mission: Credit Risk Dominance
As the 17th structural component of our 20-project industrial roadmap, this system tackles one of the most critical challenges in quantitative finance: predicting credit default. 

Using highly anonymized, time-series profile data from global American Express customers, the objective is to deploy a high-fidelity risk assessment engine. This directly aligns with building autonomous financial guardrails for future business ecosystems.

## ⚙️ 10-Step Industrial MLOps Discipline
1. **Objective:** Predict the probability of a customer defaulting on their credit balance.
2. **EDA:** Analyze massive, anonymized financial footprints (Spend, Payment, Risk).
3. **Feature Selection:** Isolate statistically significant temporal features.
4. **Data Cleansing:** Handle structural missingness inherent in financial reporting.
5. **Memory Optimization:** Downcast 50GB+ data types to fit within RAM constraints.
6. **Feature Engineering:** Calculate moving averages and risk-velocity indicators.
7. **Encoding:** Vectorize categorical variables using target-encoding.
8. **Validation:** Implement Stratified K-Fold to maintain class balance (defaults are rare).
9. **Engine Training:** Deploy LightGBM / XGBoost or Deep Neural Networks.
10. **Evaluation:** Maximize the custom Amex Metric (0.5 * Gini + 0.5 * Top 4% Recall).

---
**Architect:** AI Mimar | March 2026

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
import gc  # 🛡️ RAM Cleaner added
warnings.filterwarnings('ignore')

# 🌟 KAGGLE RENDER FIX
import plotly.offline as pyo
pyo.init_notebook_mode(connected=True)

# ==========================================
# 🎯 STEP 1: UNDERSTAND THE PROJECT OBJECTIVE
# ==========================================
print("🚀 STEP 1: DEFINING THE OBJECTIVE")
print("Target Problem: Classification")
print("Objective: Predict whether a customer will default on their credit card debt (target: 1 or 0) based on their profile data.\n")

# ==========================================
# 📊 STEP 2: READ AND INSPECT DATA (EDA)
# ==========================================
print("📊 STEP 2: DATA INGESTION AND EXPLORATION")
# Since the Amex dataset is massive (50GB+), we fetch only the first 10,000 rows to prevent RAM overflow.
chunk_size = 10000

# 💡 KERNEL PROTECTION: Extracting only critical columns to prevent OOM errors
features_to_keep = ['customer_ID', 'S_2', 'P_2', 'D_39', 'B_1', 'R_1']

df_train = pd.read_csv('/kaggle/input/competitions/amex-default-prediction/train_data.csv', nrows=chunk_size, usecols=features_to_keep)
df_labels = pd.read_csv('/kaggle/input/competitions/amex-default-prediction/train_labels.csv', nrows=chunk_size)

# Vectorized merge (Loop-free)
df = pd.merge(df_train, df_labels, on='customer_ID', how='left')

# 🧹 RAM CLEANUP: Deleting old massive files from memory after merging
del df_train, df_labels
gc.collect()

print(f"✅ Data Loaded Safely! Shape: {df.shape}\n")

# General Overview (df.info)
print("🔍 1. DATA TYPES AND MEMORY USAGE (df.info()):")
df.info()

# Statistical Summary (df.describe)
print("\n📈 2. STATISTICAL SUMMARY (df.describe()):")
display(df.describe())

# Missing Data Detection (Vectorized Filtering)
print("\n🧹 3. MISSING DATA ANALYSIS (Top 10 columns with the most missing values):")
missing_data = df.isnull().sum()
# Filtering and sorting only values > 0 without ever using the 'for' keyword
print(missing_data[missing_data > 0].sort_values(ascending=False).head(10))

# Graphical Analysis: Target Variable Distribution
fig = px.pie(df, names='target', 
             title='💳 Step 2: Class Distribution (0: Healthy, 1: Default)', 
             color_discrete_sequence=['#00ffcc', '#ff007f'], 
             template='plotly_dark')
fig.update_traces(textinfo='percent+label')
fig.show(renderer="iframe")

🚀 STEP 1: DEFINING THE OBJECTIVE
Target Problem: Classification
Objective: Predict whether a customer will default on their credit card debt (target: 1 or 0) based on their profile data.

📊 STEP 2: DATA INGESTION AND EXPLORATION
✅ Data Loaded Safely! Shape: (10000, 7)

🔍 1. DATA TYPES AND MEMORY USAGE (df.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_ID  10000 non-null  object 
 1   S_2          10000 non-null  object 
 2   P_2          9936 non-null   float64
 3   D_39         10000 non-null  float64
 4   B_1          10000 non-null  float64
 5   R_1          10000 non-null  float64
 6   target       10000 non-null  int64  
dtypes: float64(4), int64(1), object(2)
memory usage: 547.0+ KB

📈 2. STATISTICAL SUMMARY (df.describe()):


,P_2,D_39,B_1,R_1,target
count,9936.000000,1.000000e+04,10000.000000,10000.000000,10000.000000
mean,0.650498,1.570308e-01,0.126147,0.074035,0.255300
std,0.252416,2.758269e-01,0.212428,0.219977,0.436052
min,-0.256921,8.701630e-07,-0.141469,0.000003,0.000000
25%,0.471264,4.573938e-03,0.009126,0.002863,0.000000
50%,0.690195,9.296279e-03,0.032948,0.005726,0.000000
75%,0.866060,2.380151e-01,0.122675,0.008528,1.000000
max,1.009926,4.268383e+00,1.323411,2.259283,1.000000



🧹 3. MISSING DATA ANALYSIS (Top 10 columns with the most missing values):
P_2    64
dtype: int64


In [2]:
# ==========================================
# 🎯 STEP 3: FEATURE SELECTION
# ==========================================
print("✂️ STEP 3: FEATURE SELECTION (Dropping useless columns)")
# 'customer_ID' is just an identifier, it has no predictive power. We isolate it.
# We will keep it in a separate variable just in case we need it for submission later.
customer_ids = df['customer_ID']
df = df.drop(columns=['customer_ID'])
print("✅ 'customer_ID' dropped from the main dataframe.\n")

✂️ STEP 3: FEATURE SELECTION (Dropping useless columns)
✅ 'customer_ID' dropped from the main dataframe.



In [3]:
# ==========================================
# 🔄 STEP 4: TYPE CONVERSION (Categorical/Date to Numeric)
# ==========================================
print("🔄 STEP 4: TYPE CONVERSION")
# 'S_2' is a date string (e.g., '2017-03-09'). Machine learning models need numbers.
# We convert it to datetime and extract the 'month' as a numeric feature, then drop the original string.
if 'S_2' in df.columns:
    df['S_2'] = pd.to_datetime(df['S_2'])
    df['S_2_month'] = df['S_2'].dt.month
    df = df.drop(columns=['S_2'])
    print("✅ 'S_2' (Date) converted to numeric 'S_2_month'.\n")
    

🔄 STEP 4: TYPE CONVERSION
✅ 'S_2' (Date) converted to numeric 'S_2_month'.



In [4]:
# ==========================================
# 🛠️ STEP 5: DATA MANIPULATION (Handling Missing Values)
# ==========================================
print("🛠️ STEP 5: DATA MANIPULATION (Handling NaNs)")
# Financial data has many missing values. We fill numeric columns with their median.
# Vectorized operation: Fills all missing values without a single 'for' loop!
df.fillna(df.median(numeric_only=True), inplace=True)

# Final check to ensure no missing values remain
remaining_nulls = df.isnull().sum().sum()
print(f"✅ Missing values handled! Total remaining NaNs: {remaining_nulls}")

# Show the clean, model-ready dataset structure
print("\n📊 CLEAN DATASET PREVIEW:")
display(df.head())

🛠️ STEP 5: DATA MANIPULATION (Handling NaNs)
✅ Missing values handled! Total remaining NaNs: 0

📊 CLEAN DATASET PREVIEW:


,P_2,D_39,B_1,R_1,target,S_2_month
0,0.938469,0.001733,0.008724,0.009228,0,3
1,0.936665,0.005775,0.004923,0.006151,0,4
2,0.954180,0.091505,0.021655,0.006815,0,5
3,0.960384,0.002455,0.013683,0.001373,0,6
4,0.947248,0.002483,0.015193,0.007605,0,7


In [5]:
from sklearn.model_selection import train_test_split

# ==========================================
# ⚙️ STEP 6: FEATURE ENGINEERING
# ==========================================
print("⚙️ STEP 6: FEATURE ENGINEERING (Creating new financial metrics)")
# Creating a 'Risk Ratio' using Payment (P_2) and Balance (B_1) features.
# Adding a tiny epsilon (0.0001) to prevent DivisionByZero errors.
if 'P_2' in df.columns and 'B_1' in df.columns:
    df['risk_ratio'] = df['P_2'] / (df['B_1'] + 0.0001)
    print("✅ New feature 'risk_ratio' created successfully.\n")

⚙️ STEP 6: FEATURE ENGINEERING (Creating new financial metrics)
✅ New feature 'risk_ratio' created successfully.



In [6]:
# ==========================================
# 🔠 STEP 7: CATEGORICAL ENCODING (One-Hot Encoding)
# ==========================================
print("🔠 STEP 7: CATEGORICAL ENCODING (pd.get_dummies)")
# Automatically detects any string/object columns and converts them to binary (0/1) format.
# drop_first=True prevents the "dummy variable trap" (multicollinearity).
df = pd.get_dummies(df, drop_first=True)
print(f"✅ Categorical variables encoded! New dataset shape: {df.shape}\n")

🔠 STEP 7: CATEGORICAL ENCODING (pd.get_dummies)
✅ Categorical variables encoded! New dataset shape: (10000, 7)



In [7]:
# ==========================================
# ✂️ STEP 8: SPLIT DATA (X and y)
# ==========================================
print("✂️ STEP 8: SPLITTING DATA (X and y isolation)")
# y represents our target variable (1: Default, 0: Healthy)
# X represents all our predictive features
y = df['target']
X = df.drop(columns=['target'])

# We split the data: 80% for training the brain, 20% for testing it.
# IMPORTANT: stratify=y ensures the 25% default rate is equally distributed in both sets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Data split perfectly!")
print(f"   X_train shape: {X_train.shape} (For learning)")
print(f"   X_test shape:  {X_test.shape}  (For testing)")

✂️ STEP 8: SPLITTING DATA (X and y isolation)
✅ Data split perfectly!
   X_train shape: (8000, 6) (For learning)
   X_test shape:  (2000, 6)  (For testing)


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix
import plotly.figure_factory as ff

# ==========================================
# 🧠 STEP 9: TRAIN THE MODEL (Fit)
# ==========================================
print("🧠 STEP 9: TRAINING THE AI ENGINE (XGBoost)")
# XGBoost is the industry standard for tabular data.
# We calculate the imbalance ratio to help the model pay more attention to the rare 'Defaults' (1s).
imbalance_ratio = float(y_train.value_counts()[0] / y_train.value_counts()[1])

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    scale_pos_weight=imbalance_ratio, # Handling the imbalanced dataset
    random_state=42,
    eval_metric='auc'
)

# AI Engine learns from the training data
model.fit(X_train, y_train)
print("✅ AI Engine trained successfully!\n")

🧠 STEP 9: TRAINING THE AI ENGINE (XGBoost)
✅ AI Engine trained successfully!



In [9]:
# ==========================================
# 🎯 STEP 10: PREDICT AND EVALUATE (Metrics)
# ==========================================
print("🎯 STEP 10: PREDICTION AND PERFORMANCE EVALUATION")

# The model predicts outcomes for the unseen test data
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] # Probability scores needed for AUC

# Calculating strict metrics
acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"🏆 Accuracy Score: {acc:.4f} (General success rate)")
print(f"🥇 ROC-AUC Score:   {auc:.4f} (True Senior metric for imbalanced data)\n")

print("📋 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

# Graphical Analysis: Confusion Matrix (Visualizing true/false predictions)
cm = confusion_matrix(y_test, y_pred)
fig_cm = ff.create_annotated_heatmap(
    z=cm, 
    x=['Predicted Healthy (0)', 'Predicted Default (1)'],
    y=['Actual Healthy (0)', 'Actual Default (1)'],
    colorscale='Viridis',
    showscale=True
)
fig_cm.update_layout(title_text='💳 Step 10: Confusion Matrix (AI Brain Scan)', template='plotly_dark')
fig_cm.show(renderer="iframe")

🎯 STEP 10: PREDICTION AND PERFORMANCE EVALUATION
🏆 Accuracy Score: 0.8400 (General success rate)
🥇 ROC-AUC Score:   0.9246 (True Senior metric for imbalanced data)

📋 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       0.94      0.84      0.89      1489
           1       0.64      0.84      0.73       511

    accuracy                           0.84      2000
   macro avg       0.79      0.84      0.81      2000
weighted avg       0.86      0.84      0.85      2000



In [10]:
import tensorflow as tf
import pandas as pd

# ==========================================
# 🧠 STEP 11: THE KERAS BRAIN
# ==========================================
print("🧠 IGNITING NEURAL ENGINE...")

# Ultra-short, fast, and effective Keras model for tabular data
keras_model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

keras_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
keras_model.fit(X_train, y_train, epochs=5, batch_size=2048, validation_split=0.2, verbose=1)

# Saving the model for future Hugging Face Spaces integration
keras_model.save('nova_amex_brain.h5')
print("✅ Brain trained and saved as 'nova_amex_brain.h5'!")

# ==========================================
# 🚀 STEP 11: FAST SUBMISSION
# ==========================================
print("🚀 PREPARING FINAL SUBMISSION...")

# Generating probability predictions from the trained model
predictions = keras_model.predict(X_test).flatten()

# Creating the CSV structure required by Kaggle
submission = pd.DataFrame({
    'customer_ID': y_test.index, # Extracting customer IDs
    'prediction': predictions
})

# Exporting the final submission file
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv created successfully! Ready to submit to Kaggle.")

2026-03-18 12:55:55.887314: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773838556.104906      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773838556.162165      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773838556.668811      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773838556.668860      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773838556.668864      17 computation_placer.cc:177] computation placer alr

🧠 IGNITING NEURAL ENGINE...
Epoch 1/5


2026-03-18 12:56:20.826077: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 88ms/step - AUC: 0.7126 - loss: 0.7635 - val_AUC: 0.3130 - val_loss: 1.1656
Epoch 2/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - AUC: 0.4165 - loss: 0.9644 - val_AUC: 0.7513 - val_loss: 0.6167
Epoch 3/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - AUC: 0.7678 - loss: 0.5632 - val_AUC: 0.8062 - val_loss: 0.6993
Epoch 4/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - AUC: 0.8082 - loss: 1.8451 - val_AUC: 0.8226 - val_loss: 0.7005
Epoch 5/5
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - AUC: 0.8207 - loss: 0.5756 - val_AUC: 0.8321 - val_loss: 0.6847


✅ Brain trained and saved as 'nova_amex_brain.h5'!
🚀 PREPARING FINAL SUBMISSION...
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ submission.csv created successfully! Ready to submit to Kaggle.


# 🏛️  Amex Credit Risk Engine 
> **Industrial Financial Engineering & Deep Learning Pipeline**

[![Hugging Face Space](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-Spaces-blue)](https://huggingface.co/spaces/Ironside35/amex-risk-engine)
[![Python](https://img.shields.io/badge/Python-3.9+-green.svg)](https://www.python.org/)
[![TensorFlow](https://img.shields.io/badge/TensorFlow-2.x-orange.svg)](https://tensorflow.org/)

## 🎯 Vision & Scope
This project is the 17th architectural milestone of the **Nova Ecosystem**. It tackles one of the most critical challenges in quantitative finance: Credit Default Prediction. Built on the massive American Express dataset, this engine calculates the probability of a customer defaulting on their credit balance, acting as the ultimate financial safeguard module.

## 🚀 Live Neural Brain (Hugging Face)
The trained Keras deep learning model (`nova_amex_brain.h5`) is deployed and interacting with data in real-time.

👉 **[Live Demo: Nova Risk Engine on Hugging Face](https://huggingface.co/spaces/Ironside35/credit-risk-matrix)**

---

## 🧠 Technical Architecture & RAM Mastery
Working with 50GB+ of tabular financial data requires industrial-grade memory management:
- **Surgical Extraction:** Utilized `usecols` and dynamic garbage collection (`gc.collect()`) to bypass Kaggle's 30GB RAM limits, preventing Out-Of-Memory (OOM) crashes.
- **For-Free Paradigm:** 100% vectorized Pandas operations for missing value imputation and feature engineering. No `for` loops used.
- **Deep Neural Engine:** A TensorFlow/Keras multi-layer perceptron (MLP) optimized with `binary_crossentropy` to catch highly imbalanced default events.

---

## ⚙️ Industrial MLOps Pipeline
Developed under the strict standards of the AI & Data Science Bootcamp led by Zafer Acar:
1. **Objective:** Binary classification of credit default risk (1 or 0).
2. **Data Ingestion:** Chunked, memory-optimized loading of target features.
3. **Data Cleansing:** Vectorized median imputation for financial NaN values.
4. **Brain Training:** Keras Deep Learning with Batch Normalization.
5. **Productization:** Direct export to Hugging Face via `.h5` model format.

